# Cell 1 — Environment and project path

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/ML_final')

!pip install -q dagshub mlflow joblib cloudpickle

print("Working directory:", os.getcwd())

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 132.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

# Cell 2 — Imports, reproducibility, and device

In [2]:
import gc
import json
import math
import random
import tempfile
import time
import warnings
import zipfile
from copy import deepcopy
from pathlib import Path

import cloudpickle
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin
from sklearn.pipeline import Pipeline
from sklearn.utils.validation import check_is_fitted

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("Device :", DEVICE)

PyTorch: 2.11.0+cu128
Device : cuda


# Cell 3 — DagsHub and MLflow experiment setup

In [3]:
import dagshub
import mlflow

from mlflow import MlflowClient
from mlflow.entities import ViewType

dagshub.init(
    repo_owner="skupr23",
    repo_name="ml_final",
    mlflow=True,
)

EXPERIMENT_NAME = "NBEATS_Training"
REGISTERED_MODEL_NAME = "WalmartSalesForecast"

client = MlflowClient()
matches = client.search_experiments(
    view_type=ViewType.ALL,
    filter_string=f"name = '{EXPERIMENT_NAME}'",
)

if matches and matches[0].lifecycle_stage == "deleted":
    client.restore_experiment(matches[0].experiment_id)
    print("Restored deleted experiment:", EXPERIMENT_NAME)

mlflow.set_experiment(EXPERIMENT_NAME)

if mlflow.active_run() is not None:
    mlflow.end_run()

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("Tracking URI :", mlflow.get_tracking_uri())
print("Experiment   :", experiment.name)
print("Experiment ID:", experiment.experiment_id)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=7a6d1a26-c89b-4f30-bb20-54de60d0c5b1&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=0d8db0d93f18409d3b0ab8e73e5e01afd1740d2039c3dd8db641aa2a33dcda15




Accessing as skupr23

Initialized MLflow to track repo "skupr23/ml_final"

Repository skupr23/ml_final initialized!

Tracking URI : https://dagshub.com/skupr23/ml_final.mlflow
Experiment   : NBEATS_Training
Experiment ID: 6


# Cell 4 — Load the raw competition files

In [4]:
DATA_DIR = Path("data")

def ensure_csv(name):
    csv_path = DATA_DIR / f"{name}.csv"
    zip_path = DATA_DIR / f"{name}.csv.zip"

    if csv_path.exists():
        return csv_path

    if zip_path.exists():
        with zipfile.ZipFile(zip_path, "r") as archive:
            archive.extractall(DATA_DIR)
        return csv_path

    raise FileNotFoundError(
        f"Neither {csv_path} nor {zip_path} exists."
    )

train_path = ensure_csv("train")
test_path = ensure_csv("test")
features_path = ensure_csv("features")
stores_path = ensure_csv("stores")

train_raw = pd.read_csv(train_path, parse_dates=["Date"])
test_raw = pd.read_csv(test_path, parse_dates=["Date"])
features_raw = pd.read_csv(features_path, parse_dates=["Date"])
stores_raw = pd.read_csv(stores_path)

train_raw = train_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
test_raw = test_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

print("train   :", train_raw.shape)
print("test    :", test_raw.shape)
print("features:", features_raw.shape)
print("stores  :", stores_raw.shape)
print("train dates:", train_raw["Date"].min(), "to", train_raw["Date"].max())
print("test dates :", test_raw["Date"].min(), "to", test_raw["Date"].max())

train   : (421570, 5)
test    : (115064, 4)
features: (8190, 12)
stores  : (45, 3)
train dates: 2010-02-05 00:00:00 to 2012-10-26 00:00:00
test dates : 2012-11-02 00:00:00 to 2013-07-26 00:00:00


# Cell 5 — Raw-data validation

In [5]:
REQUIRED_TRAIN_COLUMNS = {
    "Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"
}
REQUIRED_TEST_COLUMNS = {
    "Store", "Dept", "Date", "IsHoliday"
}

missing_train = REQUIRED_TRAIN_COLUMNS - set(train_raw.columns)
missing_test = REQUIRED_TEST_COLUMNS - set(test_raw.columns)

if missing_train:
    raise ValueError(f"Missing train columns: {sorted(missing_train)}")
if missing_test:
    raise ValueError(f"Missing test columns: {sorted(missing_test)}")

if train_raw["Date"].isna().any() or test_raw["Date"].isna().any():
    raise ValueError("Invalid dates found.")

train_duplicates = train_raw.duplicated(["Store", "Dept", "Date"]).sum()
test_duplicates = test_raw.duplicated(["Store", "Dept", "Date"]).sum()

if train_duplicates:
    raise ValueError(f"Duplicate train Store-Dept-Date rows: {train_duplicates}")
if test_duplicates:
    raise ValueError(f"Duplicate test Store-Dept-Date rows: {test_duplicates}")

print("Raw schema is valid.")
print("Store-Dept series:", train_raw[["Store", "Dept"]].drop_duplicates().shape[0])
print("Negative sales rows:", int((train_raw["Weekly_Sales"] < 0).sum()))

Raw schema is valid.
Store-Dept series: 3331
Negative sales rows: 1285


# Cell 6 — Competition metric

In [6]:
def wmae(y_true, y_pred, is_holiday):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    is_holiday = np.asarray(is_holiday, dtype=bool)

    if len(y_true) == 0:
        return float("nan")

    weights = np.where(is_holiday, 5.0, 1.0)
    return float(
        np.sum(weights * np.abs(y_true - y_pred))
        / np.sum(weights)
    )

# Cell 7 — Chronological comparison holdout

In [7]:

HOLDOUT_WEEKS = 39

all_train_dates = np.array(
    sorted(train_raw["Date"].unique())
)

if len(all_train_dates) <= HOLDOUT_WEEKS:
    raise ValueError(
        "Not enough dates to create the requested holdout."
    )

holdout_dates = all_train_dates[-HOLDOUT_WEEKS:]
holdout_start = pd.Timestamp(holdout_dates[0])

fit_pool = train_raw[
    train_raw["Date"] < holdout_start
].copy()
holdout = train_raw[
    train_raw["Date"].isin(holdout_dates)
].copy()

if holdout["Date"].nunique() != HOLDOUT_WEEKS:
    raise RuntimeError(
        "Holdout split does not contain exactly "
        f"{HOLDOUT_WEEKS} dates."
    )

print("fit_pool ends :", fit_pool["Date"].max())
print("fit_pool dates:", fit_pool["Date"].nunique())
print("holdout range :", holdout["Date"].min(), "to", holdout["Date"].max())
print("holdout dates :", holdout["Date"].nunique())
print("fit rows      :", len(fit_pool))
print("holdout rows  :", len(holdout))


fit_pool ends : 2012-01-27 00:00:00
fit_pool dates: 104
holdout range : 2012-02-03 00:00:00 to 2012-10-26 00:00:00
holdout dates : 39
fit rows      : 305982
holdout rows  : 115588


# Cell 8 — Log cleaning and dataset lineage

In [8]:
CLEANING_STEPS = [
    {
        "step": "load_raw",
        "detail": "Read train.csv and test.csv; parse Date."
    },
    {
        "step": "validate_schema",
        "detail": "Require Store, Dept, Date, IsHoliday and Weekly_Sales in train."
    },
    {
        "step": "validate_keys",
        "detail": "Reject duplicate Store-Dept-Date rows."
    },
    {
        "step": "sort_panel",
        "detail": "Sort every series chronologically by Store, Dept, Date."
    },
    {
        "step": "weekly_window_filter",
        "detail": "Training windows are retained only when every adjacent date is exactly 7 days apart."
    },
]

with mlflow.start_run(run_name="NBEATS_Cleaning"):
    mlflow.set_tags({
        "stage": "cleaning",
        "architecture": "N-BEATS",
        "notebook": "model_experiment_NBEATS.ipynb",
        "dataset": "walmart-recruiting-store-sales-forecasting",
    })

    mlflow.log_params({
        "target": "Weekly_Sales",
        "series_keys": ["Store", "Dept"],
        "date_column": "Date",
        "holiday_weight": 5,
        "normal_weight": 1,
        "n_cleaning_steps": len(CLEANING_STEPS),
    })

    mlflow.log_metrics({
        "train_rows": len(train_raw),
        "test_rows": len(test_raw),
        "train_columns": train_raw.shape[1],
        "test_columns": test_raw.shape[1],
        "n_stores": train_raw["Store"].nunique(),
        "n_departments": train_raw["Dept"].nunique(),
        "n_store_department_series": (
            train_raw[["Store", "Dept"]].drop_duplicates().shape[0]
        ),
        "negative_sales_rows": int((train_raw["Weekly_Sales"] < 0).sum()),
        "train_duplicate_keys": int(train_duplicates),
        "test_duplicate_keys": int(test_duplicates),
    })

    mlflow.log_dict(
        {"steps": CLEANING_STEPS},
        "preprocessing/cleaning_steps.json",
    )
    mlflow.log_input(
        mlflow.data.from_pandas(
            train_raw,
            name="raw_train",
            targets="Weekly_Sales",
        ),
        context="training",
    )
    mlflow.log_input(
        mlflow.data.from_pandas(
            test_raw,
            name="raw_test",
        ),
        context="testing",
    )

print("Cleaning run logged.")

/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference tim

🏃 View run NBEATS_Cleaning at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/6/runs/42cc35dfa777468faaa23ea905daf3a5
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/6
Cleaning run logged.


# Cell 9 — Feature-selection decision for original N-BEATS

In [9]:


SELECTED_MODEL_INPUTS = ["Weekly_Sales history"]
STRUCTURAL_COLUMNS = ["Store", "Dept", "Date"]
WEIGHTING_COLUMNS = ["IsHoliday"]

REJECTED_EXOGENOUS_COLUMNS = [
    "Temperature",
    "Fuel_Price",
    "MarkDown1",
    "MarkDown2",
    "MarkDown3",
    "MarkDown4",
    "MarkDown5",
    "CPI",
    "Unemployment",
    "Type",
    "Size",
]

with mlflow.start_run(run_name="NBEATS_Feature_Selection"):
    mlflow.set_tags({
        "stage": "feature_selection",
        "architecture": "N-BEATS",
        "model_class": "univariate_global_forecaster",
    })

    mlflow.log_params({
        "selection_strategy": "architecture-constrained univariate input",
        "selected_model_inputs": SELECTED_MODEL_INPUTS,
        "structural_columns": STRUCTURAL_COLUMNS,
        "weighting_columns": WEIGHTING_COLUMNS,
        "n_rejected_exogenous_columns": len(REJECTED_EXOGENOUS_COLUMNS),
    })

    mlflow.log_dict(
        {
            "selected_model_inputs": SELECTED_MODEL_INPUTS,
            "structural_columns": STRUCTURAL_COLUMNS,
            "weighting_columns": WEIGHTING_COLUMNS,
            "rejected_exogenous_columns": REJECTED_EXOGENOUS_COLUMNS,
            "reason": (
                "Original N-BEATS is univariate. Exogenous regressors belong to "
                "NBEATSx and are excluded to keep architecture comparisons honest."
            ),
        },
        "features/selection.json",
    )

print("Feature-selection run logged.")

🏃 View run NBEATS_Feature_Selection at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/6/runs/c865ea89a08649d497720b3e9b9bdd19
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/6
Feature-selection run logged.


# Cell 10 — Seasonal-naive baseline

In [10]:
def seasonal_naive_predict(train_df, future_df, lag_weeks=52):
    history = train_df[
        ["Store", "Dept", "Date", "Weekly_Sales"]
    ].copy()

    history["Date"] = (
        history["Date"] + pd.Timedelta(weeks=lag_weeks)
    )
    history = history.rename(
        columns={"Weekly_Sales": "seasonal_prediction"}
    )

    result = future_df[
        ["Store", "Dept", "Date"]
    ].merge(
        history,
        on=["Store", "Dept", "Date"],
        how="left",
        validate="one_to_one",
    )

    pair_median = (
        train_df.groupby(["Store", "Dept"])["Weekly_Sales"]
        .median()
    )
    dept_median = (
        train_df.groupby("Dept")["Weekly_Sales"]
        .median()
    )
    global_median = float(train_df["Weekly_Sales"].median())

    missing = result["seasonal_prediction"].isna()

    if missing.any():
        pair_fallback = pd.MultiIndex.from_frame(
            result.loc[missing, ["Store", "Dept"]]
        ).map(pair_median)

        result.loc[missing, "seasonal_prediction"] = pair_fallback

    missing = result["seasonal_prediction"].isna()

    if missing.any():
        result.loc[missing, "seasonal_prediction"] = (
            result.loc[missing, "Dept"].map(dept_median)
        )

    result["seasonal_prediction"] = (
        result["seasonal_prediction"].fillna(global_median)
    )

    return result["seasonal_prediction"].to_numpy()


baseline_holdout_pred = seasonal_naive_predict(
    fit_pool,
    holdout,
    lag_weeks=52,
)

baseline_holdout_wmae = wmae(
    holdout["Weekly_Sales"],
    baseline_holdout_pred,
    holdout["IsHoliday"],
)

print("52-week seasonal-naive holdout WMAE:", baseline_holdout_wmae)

with mlflow.start_run(run_name="NBEATS_Baseline"):
    mlflow.set_tags({
        "stage": "baseline",
        "architecture": "seasonal_naive",
    })
    mlflow.log_params({
        "lag_weeks": 52,
        "fallback_1": "Store-Dept median",
        "fallback_2": "Department median",
        "fallback_3": "Global median",
        "holdout_weeks_parameter": HOLDOUT_WEEKS,
    })
    mlflow.log_metric(
        "holdout_wmae",
        baseline_holdout_wmae,
    )

52-week seasonal-naive holdout WMAE: 1804.6198961321131
🏃 View run NBEATS_Baseline at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/6/runs/7e5cd513683a4d6595fff51987eaaadd
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/6


# Cell 11 — Raw-input schema transformer

In [11]:
class RawWalmartSchema(BaseEstimator, TransformerMixin):
    REQUIRED_COLUMNS = ["Store", "Dept", "Date", "IsHoliday"]

    def fit(self, X, y=None):
        self._validate(X)
        return self

    def transform(self, X):
        self._validate(X)

        result = X.copy()
        result["Date"] = pd.to_datetime(
            result["Date"],
            errors="raise",
        )

        return result

    def _validate(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError(
                "RawWalmartSchema expects a pandas DataFrame."
            )

        missing = set(self.REQUIRED_COLUMNS) - set(X.columns)

        if missing:
            raise ValueError(
                f"Missing raw input columns: {sorted(missing)}"
            )

# Cell 12 — Generic N-BEATS network

In [12]:
class NBEATSBlock(nn.Module):
    def __init__(
        self,
        backcast_length,
        forecast_length,
        hidden_size,
        n_layers,
        dropout,
    ):
        super().__init__()

        layers = []
        input_size = backcast_length

        for _ in range(n_layers):
            layers.append(nn.Linear(input_size, hidden_size))
            layers.append(nn.ReLU())

            if dropout > 0:
                layers.append(nn.Dropout(dropout))

            input_size = hidden_size

        self.mlp = nn.Sequential(*layers)

        self.backcast_head = nn.Linear(
            hidden_size,
            backcast_length,
        )
        self.forecast_head = nn.Linear(
            hidden_size,
            forecast_length,
        )

    def forward(self, x):
        hidden = self.mlp(x)

        backcast = self.backcast_head(hidden)
        forecast = self.forecast_head(hidden)

        return backcast, forecast


class NBEATSNet(nn.Module):
    def __init__(
        self,
        backcast_length,
        forecast_length,
        hidden_size=256,
        n_stacks=2,
        blocks_per_stack=2,
        n_layers=4,
        dropout=0.0,
    ):
        super().__init__()

        self.blocks = nn.ModuleList()

        for _ in range(n_stacks):
            for _ in range(blocks_per_stack):
                self.blocks.append(
                    NBEATSBlock(
                        backcast_length=backcast_length,
                        forecast_length=forecast_length,
                        hidden_size=hidden_size,
                        n_layers=n_layers,
                        dropout=dropout,
                    )
                )

    def forward(self, x):
        residual = x
        forecast = torch.zeros(
            (x.shape[0], self.blocks[0].forecast_head.out_features),
            device=x.device,
            dtype=x.dtype,
        )

        for block in self.blocks:
            block_backcast, block_forecast = block(residual)
            residual = residual - block_backcast
            forecast = forecast + block_forecast

        return forecast

# Cell 13 — Global panel N-BEATS estimator

In [13]:
class NBEATSPanelForecaster(
    BaseEstimator,
    RegressorMixin,
):
    def __init__(
        self,
        backcast_length=52,
        forecast_length=13,
        hidden_size=256,
        n_stacks=2,
        blocks_per_stack=2,
        n_layers=4,
        dropout=0.0,
        learning_rate=1e-3,
        weight_decay=1e-5,
        batch_size=1024,
        epochs=15,
        stride=4,
        max_windows_per_series=24,
        gradient_clip=1.0,
        random_state=42,
        device=None,
        verbose=True,
    ):
        self.backcast_length = backcast_length
        self.forecast_length = forecast_length
        self.hidden_size = hidden_size
        self.n_stacks = n_stacks
        self.blocks_per_stack = blocks_per_stack
        self.n_layers = n_layers
        self.dropout = dropout
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.epochs = epochs
        self.stride = stride
        self.max_windows_per_series = max_windows_per_series
        self.gradient_clip = gradient_clip
        self.random_state = random_state
        self.device = device
        self.verbose = verbose

    @staticmethod
    def _is_weekly_contiguous(dates):
        if len(dates) <= 1:
            return True

        differences = (
            np.diff(dates)
            .astype("timedelta64[D]")
            .astype(np.int64)
        )

        return bool(np.all(differences == 7))

    def _resolved_device(self):
        if self.device is not None:
            requested = str(self.device)

            if requested.startswith("cuda") and not torch.cuda.is_available():
                return "cpu"

            return requested

        return "cuda" if torch.cuda.is_available() else "cpu"

    def _build_windows(self, panel):
        x_windows = []
        y_windows = []
        weight_windows = []

        total_candidates = 0
        rejected_non_contiguous = 0
        total_series = 0
        too_short_series = 0
        longest_series_length = 0

        required_length = (
            self.backcast_length + self.forecast_length
        )

        for _, group in panel.groupby(
            ["Store", "Dept"],
            sort=False,
        ):
            total_series += 1
            group = group.sort_values("Date")
            longest_series_length = max(
                longest_series_length,
                len(group),
            )

            values = group["Weekly_Sales"].to_numpy(
                dtype=np.float32
            )
            holidays = group["IsHoliday"].to_numpy(
                dtype=bool
            )
            dates = group["Date"].to_numpy(
                dtype="datetime64[D]"
            )

            if len(group) < required_length:
                too_short_series += 1
                continue

            candidate_ends = list(
                range(
                    self.backcast_length,
                    len(group) - self.forecast_length + 1,
                    self.stride,
                )
            )

            if (
                self.max_windows_per_series is not None
                and len(candidate_ends) > self.max_windows_per_series
            ):
                selected_positions = np.linspace(
                    0,
                    len(candidate_ends) - 1,
                    num=self.max_windows_per_series,
                    dtype=int,
                )
                candidate_ends = [
                    candidate_ends[position]
                    for position in selected_positions
                ]

            for split_index in candidate_ends:
                total_candidates += 1

                start = split_index - self.backcast_length
                stop = split_index + self.forecast_length

                if not self._is_weekly_contiguous(
                    dates[start:stop]
                ):
                    rejected_non_contiguous += 1
                    continue

                backcast = values[start:split_index]
                target = values[split_index:stop]
                future_holidays = holidays[split_index:stop]

                scale = max(
                    float(np.mean(np.abs(backcast))),
                    1.0,
                )

                x_windows.append(backcast / scale)
                y_windows.append(target / scale)
                weight_windows.append(
                    np.where(
                        future_holidays,
                        5.0,
                        1.0,
                    ).astype(np.float32)
                )

        if not x_windows:
            self.window_stats_ = {
                "accepted_windows": 0,
                "candidate_windows": int(total_candidates),
                "rejected_non_contiguous_windows": int(
                    rejected_non_contiguous
                ),
                "total_series": int(total_series),
                "too_short_series": int(too_short_series),
                "longest_series_length": int(
                    longest_series_length
                ),
                "required_series_length": int(
                    required_length
                ),
            }

            raise ValueError(
                "No valid weekly windows were created. "
                f"Each window requires {required_length} rows "
                f"({self.backcast_length} backcast + "
                f"{self.forecast_length} forecast), but the "
                f"longest available series has "
                f"{longest_series_length} rows. "
                f"Too-short series: {too_short_series}/{total_series}; "
                f"candidate windows: {total_candidates}; "
                f"non-contiguous candidates: "
                f"{rejected_non_contiguous}."
            )

        X = np.asarray(x_windows, dtype=np.float32)
        y = np.asarray(y_windows, dtype=np.float32)
        weights = np.asarray(weight_windows, dtype=np.float32)

        self.window_stats_ = {
            "accepted_windows": int(len(X)),
            "candidate_windows": int(total_candidates),
            "rejected_non_contiguous_windows": int(
                rejected_non_contiguous
            ),
            "total_series": int(total_series),
            "too_short_series": int(too_short_series),
            "longest_series_length": int(
                longest_series_length
            ),
            "required_series_length": int(
                required_length
            ),
        }

        return X, y, weights

    def _store_histories_and_fallbacks(self, panel):
        self.history_ = {}
        self.history_last_date_ = {}

        for key, group in panel.groupby(
            ["Store", "Dept"],
            sort=False,
        ):
            ordered = group.sort_values("Date")
            values = ordered["Weekly_Sales"].to_numpy(
                dtype=np.float32
            )
            tuple_key = tuple(key)
            self.history_[tuple_key] = values
            self.history_last_date_[tuple_key] = pd.Timestamp(
                ordered["Date"].iloc[-1]
            )

        self.pair_medians_ = (
            panel.groupby(["Store", "Dept"])["Weekly_Sales"]
            .median()
            .to_dict()
        )
        self.department_medians_ = (
            panel.groupby("Dept")["Weekly_Sales"]
            .median()
            .to_dict()
        )
        self.global_median_ = float(
            panel["Weekly_Sales"].median()
        )

    def fit(self, X, y):
        seed_everything(self.random_state)

        panel = X[
            ["Store", "Dept", "Date", "IsHoliday"]
        ].copy()
        panel["Date"] = pd.to_datetime(
            panel["Date"],
            errors="raise",
        )
        panel["Weekly_Sales"] = np.asarray(
            y,
            dtype=np.float32,
        )

        panel = panel.sort_values(
            ["Store", "Dept", "Date"]
        ).reset_index(drop=True)

        self._store_histories_and_fallbacks(panel)

        X_windows, y_windows, weight_windows = (
            self._build_windows(panel)
        )

        dataset = torch.utils.data.TensorDataset(
            torch.from_numpy(X_windows),
            torch.from_numpy(y_windows),
            torch.from_numpy(weight_windows),
        )

        generator = torch.Generator()
        generator.manual_seed(self.random_state)

        loader = torch.utils.data.DataLoader(
            dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
            generator=generator,
        )

        self.device_ = self._resolved_device()

        self.model_ = NBEATSNet(
            backcast_length=self.backcast_length,
            forecast_length=self.forecast_length,
            hidden_size=self.hidden_size,
            n_stacks=self.n_stacks,
            blocks_per_stack=self.blocks_per_stack,
            n_layers=self.n_layers,
            dropout=self.dropout,
        ).to(self.device_)

        optimizer = torch.optim.Adam(
            self.model_.parameters(),
            lr=self.learning_rate,
            weight_decay=self.weight_decay,
        )

        self.training_history_ = []

        for epoch in range(self.epochs):
            self.model_.train()

            weighted_error_sum = 0.0
            weight_sum = 0.0

            for batch_x, batch_y, batch_weights in loader:
                batch_x = batch_x.to(
                    self.device_,
                    non_blocking=True,
                )
                batch_y = batch_y.to(
                    self.device_,
                    non_blocking=True,
                )
                batch_weights = batch_weights.to(
                    self.device_,
                    non_blocking=True,
                )

                optimizer.zero_grad(set_to_none=True)

                prediction = self.model_(batch_x)

                absolute_error = torch.abs(
                    prediction - batch_y
                )
                loss = (
                    batch_weights * absolute_error
                ).sum() / batch_weights.sum()

                loss.backward()

                if self.gradient_clip is not None:
                    torch.nn.utils.clip_grad_norm_(
                        self.model_.parameters(),
                        self.gradient_clip,
                    )

                optimizer.step()

                weighted_error_sum += float(
                    (
                        batch_weights
                        * absolute_error.detach()
                    ).sum().cpu()
                )
                weight_sum += float(
                    batch_weights.sum().cpu()
                )

            epoch_loss = (
                weighted_error_sum / weight_sum
            )
            self.training_history_.append(epoch_loss)

            if self.verbose:
                print(
                    f"epoch {epoch + 1:02d}/{self.epochs} "
                    f"- scaled weighted MAE: {epoch_loss:.6f}"
                )

        self.model_.eval()
        self.n_features_in_ = len(
            RawWalmartSchema.REQUIRED_COLUMNS
        )

        return self

    def _fallback_history(self, store, dept):
        fallback = self.pair_medians_.get(
            (store, dept)
        )

        if fallback is None or pd.isna(fallback):
            fallback = self.department_medians_.get(dept)

        if fallback is None or pd.isna(fallback):
            fallback = self.global_median_

        return np.full(
            self.backcast_length,
            float(fallback),
            dtype=np.float32,
        )

    def _pad_history(self, history, store, dept):
        history = np.asarray(
            history,
            dtype=np.float32,
        )

        if len(history) >= self.backcast_length:
            return history[-self.backcast_length:]

        fallback = self._fallback_history(
            store,
            dept,
        )[0]

        padding = np.full(
            self.backcast_length - len(history),
            fallback,
            dtype=np.float32,
        )

        return np.concatenate([padding, history])

    def _forecast_series(self, store, dept, horizon):
        history = self.history_.get(
            (store, dept),
        )

        if history is None:
            history = self._fallback_history(
                store,
                dept,
            )
        else:
            history = history.copy()

        forecasts = []

        while len(forecasts) < horizon:
            backcast = self._pad_history(
                history,
                store,
                dept,
            )

            scale = max(
                float(np.mean(np.abs(backcast))),
                1.0,
            )

            model_input = torch.from_numpy(
                (backcast / scale).astype(np.float32)
            ).unsqueeze(0).to(self.device_)

            with torch.no_grad():
                chunk = (
                    self.model_(model_input)
                    .squeeze(0)
                    .detach()
                    .cpu()
                    .numpy()
                    * scale
                )

            remaining = horizon - len(forecasts)
            accepted = chunk[:remaining].astype(np.float32)

            forecasts.extend(accepted.tolist())
            history = np.concatenate(
                [history, accepted]
            )

        return np.asarray(
            forecasts,
            dtype=np.float32,
        )

    def predict(self, X):
        check_is_fitted(
            self,
            ["model_", "history_", "global_median_"],
        )

        frame = X[
            ["Store", "Dept", "Date", "IsHoliday"]
        ].copy()
        frame["Date"] = pd.to_datetime(
            frame["Date"],
            errors="raise",
        )
        frame["_original_order"] = np.arange(
            len(frame)
        )

        predictions = np.empty(
            len(frame),
            dtype=np.float32,
        )

        self.model_.eval()

        for (store, dept), group in frame.groupby(
            ["Store", "Dept"],
            sort=False,
        ):
            ordered = group.sort_values("Date")
            key = (store, dept)
            last_training_date = self.history_last_date_.get(key)

            if last_training_date is None:

                requested_steps = np.arange(
                    1,
                    len(ordered) + 1,
                    dtype=int,
                )
            else:
                day_offsets = (
                    ordered["Date"] - last_training_date
                ).dt.days.to_numpy()

                if np.any(day_offsets <= 0):
                    raise ValueError(
                        "Prediction dates must be later than each "
                        "series' final training date."
                    )

                if np.any(day_offsets % 7 != 0):
                    raise ValueError(
                        "Prediction dates must lie on the weekly grid."
                    )

                requested_steps = (
                    day_offsets // 7
                ).astype(int)

            complete_forecast = self._forecast_series(
                store=store,
                dept=dept,
                horizon=int(requested_steps.max()),
            )

            group_prediction = complete_forecast[
                requested_steps - 1
            ]

            predictions[
                ordered["_original_order"].to_numpy()
            ] = group_prediction

        return predictions

    def prepare_for_serialization(self):
        check_is_fitted(self, ["model_"])

        self.model_ = self.model_.to("cpu")
        self.device_ = "cpu"
        self.model_.eval()

        return self

    def __getstate__(self):
        state = self.__dict__.copy()
        model = state.pop("model_", None)

        if model is not None:
            model = model.to("cpu")
            state["_serialized_model_state_"] = {
                name: tensor.detach().cpu()
                for name, tensor in model.state_dict().items()
            }

        state["device_"] = "cpu"
        return state

    def __setstate__(self, state):
        model_state = state.pop(
            "_serialized_model_state_",
            None,
        )
        self.__dict__.update(state)

        if model_state is not None:
            self.model_ = NBEATSNet(
                backcast_length=self.backcast_length,
                forecast_length=self.forecast_length,
                hidden_size=self.hidden_size,
                n_stacks=self.n_stacks,
                blocks_per_stack=self.blocks_per_stack,
                n_layers=self.n_layers,
                dropout=self.dropout,
            )
            self.model_.load_state_dict(model_state)
            self.model_.to("cpu")
            self.model_.eval()
            self.device_ = "cpu"


# Cell 14 — Pipeline factory

In [14]:
def make_nbeats_pipeline(config):
    return Pipeline([
        (
            "raw_schema",
            RawWalmartSchema(),
        ),
        (
            "model",
            NBEATSPanelForecaster(**config),
        ),
    ])

# Cell 15 — Expanding-window cross-validation folds

In [15]:
CV_FOLDS = 3
CV_VALIDATION_WEEKS = 13
CV_BACKCAST_WEEKS = 52
MIN_CV_TRAIN_WEEKS = (
    CV_BACKCAST_WEEKS + CV_VALIDATION_WEEKS
)

def make_expanding_folds(
    data,
    n_folds=CV_FOLDS,
    validation_weeks=CV_VALIDATION_WEEKS,
    minimum_train_weeks=MIN_CV_TRAIN_WEEKS,
):
    unique_dates = np.array(
        sorted(data["Date"].unique())
    )

    required_dates = (
        minimum_train_weeks
        + n_folds * validation_weeks
    )

    if len(unique_dates) < required_dates:
        raise ValueError(
            "Not enough dates for the requested expanding folds. "
            f"Need at least {required_dates} dates: "
            f"{minimum_train_weeks} initial training weeks + "
            f"{n_folds} × {validation_weeks} validation weeks. "
            f"Available: {len(unique_dates)}."
        )

    folds = []

    for fold_index in range(n_folds):
        validation_end = (
            len(unique_dates)
            - (n_folds - fold_index - 1)
            * validation_weeks
        )
        validation_start = (
            validation_end - validation_weeks
        )

        train_dates = unique_dates[:validation_start]
        validation_dates = unique_dates[
            validation_start:validation_end
        ]

        if len(train_dates) < minimum_train_weeks:
            raise RuntimeError(
                f"Fold {fold_index + 1} has only "
                f"{len(train_dates)} training weeks; "
                f"{minimum_train_weeks} are required."
            )

        fold_train = data[
            data["Date"].isin(train_dates)
        ].copy()
        fold_validation = data[
            data["Date"].isin(validation_dates)
        ].copy()

        folds.append({
            "fold": fold_index + 1,
            "train": fold_train,
            "validation": fold_validation,
            "train_weeks": len(train_dates),
            "validation_weeks": len(validation_dates),
            "train_end": pd.Timestamp(train_dates[-1]),
            "validation_start": pd.Timestamp(
                validation_dates[0]
            ),
            "validation_end": pd.Timestamp(
                validation_dates[-1]
            ),
        })

    return folds


folds = make_expanding_folds(fit_pool)

for fold in folds:
    print(
        f"fold {fold['fold']}: "
        f"{fold['train_weeks']} training weeks through "
        f"{fold['train_end'].date()} | "
        f"validation {fold['validation_start'].date()} "
        f"to {fold['validation_end'].date()} | "
        f"{len(fold['train']):,} / "
        f"{len(fold['validation']):,} rows"
    )


fold 1: 65 training weeks through 2011-04-29 | validation 2011-05-06 to 2011-07-29 | 190,674 / 38,164 rows
fold 2: 78 training weeks through 2011-07-29 | validation 2011-08-05 to 2011-10-28 | 228,838 / 38,346 rows
fold 3: 91 training weeks through 2011-10-28 | validation 2011-11-04 to 2012-01-27 | 267,184 / 38,798 rows


# Cell 16 — Hyperparameter configurations

In [16]:


QUICK_MODE = False

if QUICK_MODE:
    SEARCH_CONFIGS = {
        "small_underfit": {
            "backcast_length": 52,
            "forecast_length": 13,
            "hidden_size": 128,
            "n_stacks": 1,
            "blocks_per_stack": 1,
            "n_layers": 2,
            "dropout": 0.0,
            "learning_rate": 1e-3,
            "weight_decay": 1e-5,
            "batch_size": 1024,
            "epochs": 5,
            "stride": 6,
            "max_windows_per_series": 12,
            "gradient_clip": 1.0,
            "random_state": SEED,
            "device": DEVICE,
            "verbose": False,
        },
        "balanced": {
            "backcast_length": 52,
            "forecast_length": 13,
            "hidden_size": 256,
            "n_stacks": 2,
            "blocks_per_stack": 2,
            "n_layers": 4,
            "dropout": 0.0,
            "learning_rate": 1e-3,
            "weight_decay": 1e-5,
            "batch_size": 1024,
            "epochs": 8,
            "stride": 4,
            "max_windows_per_series": 16,
            "gradient_clip": 1.0,
            "random_state": SEED,
            "device": DEVICE,
            "verbose": False,
        },
    }
else:
    SEARCH_CONFIGS = {
        "small_underfit": {
            "backcast_length": 52,
            "forecast_length": 13,
            "hidden_size": 128,
            "n_stacks": 1,
            "blocks_per_stack": 1,
            "n_layers": 2,
            "dropout": 0.0,
            "learning_rate": 1e-3,
            "weight_decay": 1e-5,
            "batch_size": 1024,
            "epochs": 8,
            "stride": 6,
            "max_windows_per_series": 16,
            "gradient_clip": 1.0,
            "random_state": SEED,
            "device": DEVICE,
            "verbose": False,
        },
        "balanced": {
            "backcast_length": 52,
            "forecast_length": 13,
            "hidden_size": 256,
            "n_stacks": 2,
            "blocks_per_stack": 2,
            "n_layers": 4,
            "dropout": 0.0,
            "learning_rate": 8e-4,
            "weight_decay": 1e-5,
            "batch_size": 1024,
            "epochs": 15,
            "stride": 4,
            "max_windows_per_series": 24,
            "gradient_clip": 1.0,
            "random_state": SEED,
            "device": DEVICE,
            "verbose": False,
        },
        "deep_regularized": {
            "backcast_length": 52,
            "forecast_length": 13,
            "hidden_size": 384,
            "n_stacks": 3,
            "blocks_per_stack": 2,
            "n_layers": 4,
            "dropout": 0.1,
            "learning_rate": 5e-4,
            "weight_decay": 1e-4,
            "batch_size": 1024,
            "epochs": 20,
            "stride": 4,
            "max_windows_per_series": 24,
            "gradient_clip": 1.0,
            "random_state": SEED,
            "device": DEVICE,
            "verbose": False,
        },
    }

print("Configurations:", list(SEARCH_CONFIGS))

Configurations: ['small_underfit', 'balanced', 'deep_regularized']


# Cell 17 — Cross-validation evaluator with one MLflow run per configuration

In [17]:
def evaluate_config(
    config_name,
    config,
    folds,
):
    fold_rows = []

    with mlflow.start_run(
        run_name=f"NBEATS_CV_{config_name}"
    ):
        mlflow.set_tags({
            "stage": "cross_validation",
            "architecture": "N-BEATS",
            "configuration": config_name,
            "validation_type": "expanding_window",
        })

        mlflow.log_params({
            **config,
            "n_cv_folds": len(folds),
            "cv_validation_weeks": CV_VALIDATION_WEEKS,
            "minimum_cv_train_weeks": MIN_CV_TRAIN_WEEKS,
            "quick_mode": QUICK_MODE,
        })

        for fold in folds:
            fold_number = fold["fold"]

            train_fold = fold["train"]
            validation_fold = fold["validation"]

            pipeline = make_nbeats_pipeline(config)

            start_time = time.perf_counter()

            pipeline.fit(
                train_fold.drop(
                    columns="Weekly_Sales"
                ),
                train_fold["Weekly_Sales"],
            )

            predictions = pipeline.predict(
                validation_fold.drop(
                    columns="Weekly_Sales"
                )
            )

            elapsed = time.perf_counter() - start_time

            fold_wmae = wmae(
                validation_fold["Weekly_Sales"],
                predictions,
                validation_fold["IsHoliday"],
            )

            holiday_mask = (
                validation_fold["IsHoliday"].to_numpy(
                    dtype=bool
                )
            )

            holiday_wmae = wmae(
                validation_fold.loc[
                    holiday_mask,
                    "Weekly_Sales",
                ],
                predictions[holiday_mask],
                validation_fold.loc[
                    holiday_mask,
                    "IsHoliday",
                ],
            )

            nonholiday_wmae = wmae(
                validation_fold.loc[
                    ~holiday_mask,
                    "Weekly_Sales",
                ],
                predictions[~holiday_mask],
                validation_fold.loc[
                    ~holiday_mask,
                    "IsHoliday",
                ],
            )

            estimator = pipeline.named_steps["model"]

            row = {
                "configuration": config_name,
                "fold": fold_number,
                "train_end": fold["train_end"],
                "validation_start": (
                    fold["validation_start"]
                ),
                "validation_end": (
                    fold["validation_end"]
                ),
                "train_rows": len(train_fold),
                "validation_rows": len(validation_fold),
                "train_weeks": fold["train_weeks"],
                "validation_weeks": fold["validation_weeks"],
                "wmae": fold_wmae,
                "holiday_wmae": holiday_wmae,
                "nonholiday_wmae": nonholiday_wmae,
                "elapsed_seconds": elapsed,
                "accepted_windows": (
                    estimator.window_stats_[
                        "accepted_windows"
                    ]
                ),
                "rejected_non_contiguous_windows": (
                    estimator.window_stats_[
                        "rejected_non_contiguous_windows"
                    ]
                ),
                "final_scaled_training_loss": (
                    estimator.training_history_[-1]
                ),
            }
            fold_rows.append(row)

            fold_metrics = {
                f"fold_{fold_number}_wmae": fold_wmae,
                f"fold_{fold_number}_holiday_wmae": holiday_wmae,
                f"fold_{fold_number}_nonholiday_wmae": nonholiday_wmae,
                f"fold_{fold_number}_elapsed_seconds": elapsed,
                f"fold_{fold_number}_accepted_windows": (
                    estimator.window_stats_[
                        "accepted_windows"
                    ]
                ),
                f"fold_{fold_number}_final_scaled_training_loss": (
                    estimator.training_history_[-1]
                ),
            }
            mlflow.log_metrics({
                name: value
                for name, value in fold_metrics.items()
                if np.isfinite(value)
            })

            for epoch, loss in enumerate(
                estimator.training_history_,
                start=1,
            ):
                mlflow.log_metric(
                    f"fold_{fold_number}_scaled_train_wmae",
                    loss,
                    step=epoch,
                )

            del pipeline, predictions, estimator
            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        results = pd.DataFrame(fold_rows)

        mlflow.log_metrics({
            "cv_mean_wmae": float(
                results["wmae"].mean()
            ),
            "cv_std_wmae": float(
                results["wmae"].std(ddof=0)
            ),
            "cv_mean_holiday_wmae": float(
                results["holiday_wmae"].mean()
            ),
            "cv_mean_nonholiday_wmae": float(
                results["nonholiday_wmae"].mean()
            ),
        })

        with tempfile.TemporaryDirectory() as tmp:
            path = os.path.join(
                tmp,
                f"{config_name}_fold_results.csv",
            )
            results.to_csv(path, index=False)
            mlflow.log_artifact(
                path,
                artifact_path="cross_validation",
            )

    return results

# Cell 18 — Run the N-BEATS cross-validation search

In [18]:
all_cv_results = []

for config_name, config in SEARCH_CONFIGS.items():
    print("=" * 80)
    print("Configuration:", config_name)
    print("=" * 80)

    config_results = evaluate_config(
        config_name=config_name,
        config=config,
        folds=folds,
    )

    all_cv_results.append(config_results)

cv_results = pd.concat(
    all_cv_results,
    ignore_index=True,
)

cv_summary = (
    cv_results.groupby("configuration")
    .agg(
        mean_wmae=("wmae", "mean"),
        std_wmae=("wmae", "std"),
        mean_holiday_wmae=(
            "holiday_wmae",
            "mean",
        ),
        mean_nonholiday_wmae=(
            "nonholiday_wmae",
            "mean",
        ),
        mean_elapsed_seconds=(
            "elapsed_seconds",
            "mean",
        ),
        mean_final_scaled_training_loss=(
            "final_scaled_training_loss",
            "mean",
        ),
    )
    .sort_values("mean_wmae")
    .reset_index()
)

display(cv_summary)

Configuration: small_underfit
🏃 View run NBEATS_CV_small_underfit at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/6/runs/1b684f3e5ce44eda806c031e33decd90
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/6
Configuration: balanced
🏃 View run NBEATS_CV_balanced at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/6/runs/f3d79a618b854fb9bb44bda2523e2007
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/6
Configuration: deep_regularized
🏃 View run NBEATS_CV_deep_regularized at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/6/runs/9ed820aa05e7403aafd95b823b2743aa
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/6


,configuration,mean_wmae,std_wmae,mean_holiday_wmae,mean_nonholiday_wmae,mean_elapsed_seconds,mean_final_scaled_training_loss
0,balanced,2543.290946,1097.742752,3423.714231,2152.583128,12.660721,0.140602
1,deep_regularized,2603.732861,1157.839823,3661.674289,2168.816288,17.423092,0.139662
2,small_underfit,3057.613809,1134.155842,3547.690075,2759.023182,10.220574,0.182137


# Cell 19 — Select the best configuration and log the CV summary

In [19]:
best_config_name = cv_summary.iloc[0][
    "configuration"
]
best_config = deepcopy(
    SEARCH_CONFIGS[best_config_name]
)
best_cv_mean_wmae = float(
    cv_summary.iloc[0]["mean_wmae"]
)
best_cv_std_wmae = float(
    cv_summary.iloc[0]["std_wmae"]
)

print("Best configuration:", best_config_name)
print("CV mean WMAE       :", best_cv_mean_wmae)
print("CV std WMAE        :", best_cv_std_wmae)
print("Parameters          :", best_config)

with mlflow.start_run(
    run_name="NBEATS_Cross_Validation_Summary"
):
    mlflow.set_tags({
        "stage": "cross_validation_summary",
        "architecture": "N-BEATS",
        "selection_metric": "mean expanding-window WMAE",
    })

    mlflow.log_params({
        "best_configuration": best_config_name,
        "n_configurations": len(SEARCH_CONFIGS),
        "n_folds": len(folds),
        "validation_weeks_per_fold": (
            CV_VALIDATION_WEEKS
        ),
    })

    mlflow.log_metrics({
        "best_cv_mean_wmae": best_cv_mean_wmae,
        "best_cv_std_wmae": best_cv_std_wmae,
    })

    mlflow.log_dict(
        best_config,
        "best_configuration/parameters.json",
    )

    with tempfile.TemporaryDirectory() as tmp:
        summary_path = os.path.join(
            tmp,
            "cv_summary.csv",
        )
        detail_path = os.path.join(
            tmp,
            "cv_fold_results.csv",
        )

        cv_summary.to_csv(
            summary_path,
            index=False,
        )
        cv_results.to_csv(
            detail_path,
            index=False,
        )

        mlflow.log_artifact(
            summary_path,
            artifact_path="cross_validation",
        )
        mlflow.log_artifact(
            detail_path,
            artifact_path="cross_validation",
        )

Best configuration: balanced
CV mean WMAE       : 2543.290945907114
CV std WMAE        : 1097.7427516782336
Parameters          : {'backcast_length': 52, 'forecast_length': 13, 'hidden_size': 256, 'n_stacks': 2, 'blocks_per_stack': 2, 'n_layers': 4, 'dropout': 0.0, 'learning_rate': 0.0008, 'weight_decay': 1e-05, 'batch_size': 1024, 'epochs': 15, 'stride': 4, 'max_windows_per_series': 24, 'gradient_clip': 1.0, 'random_state': 42, 'device': 'cuda', 'verbose': False}
🏃 View run NBEATS_Cross_Validation_Summary at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/6/runs/db31c1e87cda42b88b4c2f960a04ed7f
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/6


# Cell 20 — Honest 39-week holdout evaluation

In [20]:
validation_pipeline = make_nbeats_pipeline(
    best_config
)

validation_pipeline.fit(
    fit_pool.drop(columns="Weekly_Sales"),
    fit_pool["Weekly_Sales"],
)

holdout_prediction = validation_pipeline.predict(
    holdout.drop(columns="Weekly_Sales")
)

holdout_wmae = wmae(
    holdout["Weekly_Sales"],
    holdout_prediction,
    holdout["IsHoliday"],
)

holdout_holiday_mask = (
    holdout["IsHoliday"].to_numpy(dtype=bool)
)

holdout_holiday_wmae = wmae(
    holdout.loc[
        holdout_holiday_mask,
        "Weekly_Sales",
    ],
    holdout_prediction[holdout_holiday_mask],
    holdout.loc[
        holdout_holiday_mask,
        "IsHoliday",
    ],
)

holdout_nonholiday_wmae = wmae(
    holdout.loc[
        ~holdout_holiday_mask,
        "Weekly_Sales",
    ],
    holdout_prediction[~holdout_holiday_mask],
    holdout.loc[
        ~holdout_holiday_mask,
        "IsHoliday",
    ],
)

print("Seasonal baseline WMAE :", baseline_holdout_wmae)
print("N-BEATS holdout WMAE   :", holdout_wmae)
print("Holiday WMAE           :", holdout_holiday_wmae)
print("Non-holiday WMAE       :", holdout_nonholiday_wmae)
print(
    "Improvement over baseline:",
    baseline_holdout_wmae - holdout_wmae,
)

Seasonal baseline WMAE : 1804.6198961321131
N-BEATS holdout WMAE   : 1758.1335919404053
Holiday WMAE           : 1769.7557176886714
Non-holiday WMAE       : 1754.9704560294065
Improvement over baseline: 46.48630419170786


# Cell 21 — Log holdout validation

In [21]:
with mlflow.start_run(
    run_name="NBEATS_Validation"
):
    mlflow.set_tags({
        "stage": "validation",
        "architecture": "N-BEATS",
        "configuration": best_config_name,
        "comparison_split": (
            "same 39-week chronological rule "
            "as LightGBM notebook"
        ),
    })

    mlflow.log_params({
        **best_config,
        "fit_end": str(
            fit_pool["Date"].max().date()
        ),
        "holdout_start": str(
            holdout["Date"].min().date()
        ),
        "holdout_end": str(
            holdout["Date"].max().date()
        ),
        "holdout_unique_weeks": (
            holdout["Date"].nunique()
        ),
        "recursive_chunk_length": (
            best_config["forecast_length"]
        ),
    })

    mlflow.log_metrics({
        "baseline_holdout_wmae": (
            baseline_holdout_wmae
        ),
        "nbeats_holdout_wmae": holdout_wmae,
        "holdout_holiday_wmae": (
            holdout_holiday_wmae
        ),
        "holdout_nonholiday_wmae": (
            holdout_nonholiday_wmae
        ),
        "improvement_over_baseline": (
            baseline_holdout_wmae
            - holdout_wmae
        ),
        "cv_mean_wmae": best_cv_mean_wmae,
        "cv_std_wmae": best_cv_std_wmae,
    })

    validation_report = holdout[
        [
            "Store",
            "Dept",
            "Date",
            "Weekly_Sales",
            "IsHoliday",
        ]
    ].copy()
    validation_report["Prediction"] = (
        holdout_prediction
    )
    validation_report["Absolute_Error"] = np.abs(
        validation_report["Weekly_Sales"]
        - validation_report["Prediction"]
    )

    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(
            tmp,
            "holdout_predictions.csv",
        )
        validation_report.to_csv(
            path,
            index=False,
        )
        mlflow.log_artifact(
            path,
            artifact_path="validation",
        )

🏃 View run NBEATS_Validation at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/6/runs/7b6df14293f9479897191dba00734c16
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/6


# Cell 22 — Train the final raw-input pipeline on all training data

In [22]:
final_pipeline = make_nbeats_pipeline(
    best_config
)

final_pipeline.fit(
    train_raw.drop(columns="Weekly_Sales"),
    train_raw["Weekly_Sales"],
)

test_prediction = final_pipeline.predict(
    test_raw
)

if len(test_prediction) != len(test_raw):
    raise AssertionError(
        "Prediction length does not match test rows."
    )

if not np.isfinite(test_prediction).all():
    raise AssertionError(
        "Final test predictions contain NaN or infinity."
    )

print("Final pipeline trained.")
print("Test predictions:", test_prediction.shape)
print(
    "Prediction range:",
    float(test_prediction.min()),
    "to",
    float(test_prediction.max()),
)

Final pipeline trained.
Test predictions: (115064,)
Prediction range: -96.58174133300781 to 220451.484375


# Cell 23 — Save and reload the complete pipeline locally

In [23]:
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

PIPELINE_PATH = (
    MODELS_DIR / "nbeats_raw_input_pipeline.pkl"
)

final_pipeline.named_steps[
    "model"
].prepare_for_serialization()

with open(PIPELINE_PATH, "wb") as file:
    cloudpickle.dump(
        final_pipeline,
        file,
    )

with open(PIPELINE_PATH, "rb") as file:
    reloaded_pipeline = cloudpickle.load(file)

reloaded_prediction = reloaded_pipeline.predict(
    test_raw
)

pipeline_reload_max_abs_diff = float(
    np.max(
        np.abs(
            test_prediction
            - reloaded_prediction
        )
    )
)

print("Saved pipeline:", PIPELINE_PATH)
print(
    "Max |original - reloaded|:",
    pipeline_reload_max_abs_diff,
)

assert np.allclose(
    test_prediction,
    reloaded_prediction,
    rtol=1e-5,
    atol=5e-2,
), "Reloaded CPU pipeline changed predictions materially."

Saved pipeline: models/nbeats_raw_input_pipeline.pkl
Max |original - reloaded|: 0.046875


# Cell 24 — Log the final pipeline to MLflow

In [25]:
import mlflow.sklearn

from mlflow.models import infer_signature

input_example = test_raw.head(20).copy()
output_example = final_pipeline.predict(
    input_example
)

signature = infer_signature(
    input_example,
    output_example,
)

with mlflow.start_run(
    run_name="NBEATS_Final_Model"
) as final_run:
    mlflow.set_tags({
        "stage": "final_model",
        "architecture": "N-BEATS",
        "configuration": best_config_name,
        "input_contract": "raw test.csv rows",
        "pipeline_type": "sklearn Pipeline",
    })

    mlflow.log_params({
        **best_config,
        "selected_configuration": (
            best_config_name
        ),
        "model_input": (
            "Weekly_Sales backcast only"
        ),
        "series_keys": ["Store", "Dept"],
        "raw_required_columns": (
            RawWalmartSchema.REQUIRED_COLUMNS
        ),
    })

    mlflow.log_metrics({
        "comparison_holdout_wmae": (
            holdout_wmae
        ),
        "comparison_baseline_wmae": (
            baseline_holdout_wmae
        ),
        "cv_mean_wmae": (
            best_cv_mean_wmae
        ),
        "cv_std_wmae": (
            best_cv_std_wmae
        ),
        "pipeline_reload_max_abs_diff": (
            pipeline_reload_max_abs_diff
        ),
        "test_prediction_rows": (
            len(test_prediction)
        ),
    })

    model_info = mlflow.sklearn.log_model(
        final_pipeline,
        artifact_path="pipeline",
        serialization_format=(
            mlflow.sklearn
            .SERIALIZATION_FORMAT_CLOUDPICKLE
        ),
        input_example=input_example,
        signature=signature,
        pip_requirements=[
            f"mlflow=={mlflow.__version__}",
            f"torch=={torch.__version__.split('+')[0]}",
            f"scikit-learn",
            f"pandas",
            f"numpy",
            f"joblib",
            f"cloudpickle",
        ],
    )

    mlflow.log_artifact(
        str(PIPELINE_PATH),
        artifact_path="pipeline_backup",
    )

    mlflow.log_dict(
        {
            "best_configuration": (
                best_config_name
            ),
            "parameters": best_config,
            "selected_inputs": (
                SELECTED_MODEL_INPUTS
            ),
            "structural_columns": (
                STRUCTURAL_COLUMNS
            ),
            "weighting_columns": (
                WEIGHTING_COLUMNS
            ),
        },
        "model/model_card.json",
    )

    FINAL_RUN_ID = final_run.info.run_id
    FINAL_MODEL_URI = (
        f"runs:/{FINAL_RUN_ID}/pipeline"
    )

print("Final run ID :", FINAL_RUN_ID)
print("Model URI    :", FINAL_MODEL_URI)

/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/07/12 11:53:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 11:53:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution becaus

🏃 View run NBEATS_Final_Model at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/6/runs/62e7602a9f8e42f7a24254cc294e49c8
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/6
Final run ID : 62e7602a9f8e42f7a24254cc294e49c8
Model URI    : runs:/62e7602a9f8e42f7a24254cc294e49c8/pipeline


# Cell 25 — Register only if N-BEATS is the best architecture overall

In [26]:


REGISTER_AS_GLOBAL_BEST = False

if REGISTER_AS_GLOBAL_BEST:
    registered_version = mlflow.register_model(
        model_uri=FINAL_MODEL_URI,
        name=REGISTERED_MODEL_NAME,
    )

    print(
        "Registered model:",
        REGISTERED_MODEL_NAME,
        "| version:",
        registered_version.version,
    )
else:
    print(
        "Pipeline is logged and saved, but not registered. "
        "Register it only if N-BEATS is the best architecture overall."
    )

Pipeline is logged and saved, but not registered. Register it only if N-BEATS is the best architecture overall.


# Cell 26 — Final experiment summary

In [27]:
summary = pd.DataFrame([
    {
        "architecture": "N-BEATS",
        "best_configuration": best_config_name,
        "cv_mean_wmae": best_cv_mean_wmae,
        "cv_std_wmae": best_cv_std_wmae,
        "comparison_holdout_wmae": holdout_wmae,
        "seasonal_baseline_wmae": baseline_holdout_wmae,
        "beats_baseline": (
            holdout_wmae
            < baseline_holdout_wmae
        ),
        "pipeline_accepts_raw_test": True,
        "pipeline_reload_max_abs_diff": (
            pipeline_reload_max_abs_diff
        ),
        "registered_as_global_best": (
            REGISTER_AS_GLOBAL_BEST
        ),
        "mlflow_run_id": FINAL_RUN_ID,
    }
])

display(summary)

,architecture,best_configuration,cv_mean_wmae,cv_std_wmae,comparison_holdout_wmae,seasonal_baseline_wmae,beats_baseline,pipeline_accepts_raw_test,pipeline_reload_max_abs_diff,registered_as_global_best,mlflow_run_id
0,N-BEATS,balanced,2543.290946,1097.742752,1758.133592,1804.619896,True,True,0.046875,False,62e7602a9f8e42f7a24254cc294e49c8
